In [1]:
## 导入相关工具类
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd


In [2]:
"""## 定义数据集(Dataset)类 用于数据读取与处理"""

class Dataset_ETT_minute(Dataset):
    def __init__(self, data_path, context_points, target_points, split='train'):
        # 确保划分类型正确
        assert split in ['train', 'test', 'val']
        type_map = {'train': 0, 'val': 1, 'test': 2}
        self.set_type = type_map[split]

        # 设置预测长度和序列长度
        self.pred_len = target_points
        self.seq_len = context_points

        # 读取原始数据
        df_raw = pd.read_csv(data_path)

        # 定义不同数据集的边界索引
        border1s = [0, 12 * 30 * 24 * 4 - self.seq_len, 12 * 30 * 24 * 4 + 4 * 30 * 24 * 4 - self.seq_len]
        border2s = [12 * 30 * 24 * 4, 12 * 30 * 24 * 4 + 4 * 30 * 24 * 4, 12 * 30 * 24 * 4 + 8 * 30 * 24 * 4]

        # 根据划分类型选择相应的边界
        border1 = border1s[self.set_type]
        border2 = border2s[self.set_type]

        # 选择数据列，排除第一列（通常是时间戳）
        cols_data = df_raw.columns[1:]
        df_data = df_raw[cols_data]

        # 获取训练数据部分并进行标准化
        self.scaler = StandardScaler()
        train_data = df_data[border1s[0]:border2s[0]]
        self.scaler.fit(train_data.values)
        data = self.scaler.transform(df_data.values)

        # 根据边界索引切分输入和目标数据
        self.data_x = data[border1:border2]
        self.data_y = data[border1:border2]

    def __getitem__(self, index):
        # 计算输入序列的起始和结束位置
        s_begin = index
        s_end = s_begin + self.seq_len
        # 计算预测序列的起始和结束位置
        r_begin = s_end
        r_end = r_begin + self.pred_len

        # 获取输入序列和目标序列
        seq_x = self.data_x[s_begin:s_end]
        seq_y = self.data_y[s_end:r_end]

        # 将数据转换为浮点型张量
        return torch.from_numpy(seq_x).float(), torch.from_numpy(seq_y).float()

    def __len__(self):
        # 返回数据集的长度，确保不会超出边界
        return len(self.data_x) - self.seq_len - self.pred_len + 1
    
    
def convert_loader_to_arrays(loader, context_points, target_points):
    features = []
    labels = []
    
    for batch_x, batch_y in loader:
        # batch_x: [batch_size, context_points, channels]
        # batch_y: [batch_size, target_points, channels]
        batch_x, batch_y = batch_x.cpu().numpy(), batch_y.cpu().numpy()
        features.append(batch_x)
        labels.append(batch_y)
    
    # 合并所有batch并转换形状
    features = np.concatenate(features, axis=0)  # [data_size, context_points, channels]
    features = np.transpose(features, (0, 2, 1)).reshape(-1, context_points) # [data_size*channels, context_points]
    
    labels = np.concatenate(labels, axis=0)  # [data_size, target_points, channels]
    labels = np.transpose(labels, (0, 2, 1)).reshape(-1, target_points) # [data_size*channels, target_points]
    
    return features, labels
    

In [3]:
"""## 配置数据参数"""
# 指定数据集的路径
data_path = './data/ETTm2.csv'
# 定义输入序列的长度（回望步长）
context_points = 96
# 定义输出序列的长度（预测步长）
target_points = 96
# 定义每个批次的样本数量
batch_size = 64
# 检查是否有可用的GPU，如果有则使用GPU，否则使用CPU
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

In [4]:
"""## 加载数据"""
# 获取数据加载器
# 创建训练集、验证集和测试集的Dataset实例
train_dataset = Dataset_ETT_minute(data_path, context_points, target_points, 'train')
val_dataset = Dataset_ETT_minute(data_path, context_points, target_points, 'val')
test_dataset = Dataset_ETT_minute(data_path, context_points, target_points, 'test')

# 使用DataLoader将Dataset封装为可迭代的数据加载器
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size) 
val_loader = DataLoader(val_dataset, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_dataset, shuffle=True, batch_size=batch_size)

# 由于LinearRegression无法实现多元时间序列预测，
# 需分别将features和labels转换成形状为 [data_size*channels, context_points], [data_size*channels, target_points]的array
# 实现通道独立的时间序列预测
train_features, train_labels = convert_loader_to_arrays(train_loader, context_points, target_points)
test_features, test_labels = convert_loader_to_arrays(test_loader, context_points, target_points)
# print(train_features.shape, train_labels.shape)
# print(test_features.shape, test_labels.shape)

In [5]:
"""## 加载模型、损失函数与优化器"""
from sklearn.linear_model import LinearRegression
model = LinearRegression() # 可以在这里替换你想使用的模型

model.fit(train_features, train_labels)

# 查看系数和截距
# print("Coefficients:", model.coef_.shape)  
# print("Intercept:", model.intercept_.shape)  

# 预测测试集
test_pred = model.predict(test_features)


# 计算平均绝对误差 (MAE)
mae = mean_absolute_error(test_pred, test_labels)

# 计算均方误差 (MSE)
mse = mean_squared_error(test_pred, test_labels)

# 打印测试集的MAE和MSE
print(f"Test MAE: {mae:.4f}, Test MSE: {mse:.4f}")


Test MAE: 0.2973, Test MSE: 0.1979
